# 실습 6 도로 균열 유무를 분류하는 합성곱 신경망 모델 구현하기 (학생용)

교과서 149~153쪽

**문제 상황**: 폭염과 폭우로 도로에 금이 가는 균열이 생기면 큰 사고로 이어질 수 있다. 사진으로 도로 균열을 자동으로 찾아낼 수 있을까?

[단계 1] 문제 정의하기: 사전 훈련된 합성곱 신경망 모델 **VGG16**으로 도로의 균열 유무를 판단해 보자.

**준비물**: 교과서 149쪽 링크에서 내려받은 `crack.zip` (train 300장, test 80장 / Negative: 균열 없음, Positive: 균열 있음)

**사용 방법**
- 위에서부터 차례로 ▶ 버튼을 눌러 실행합니다. 오른쪽의 선택 상자와 슬라이더로 설정을 바꿉니다.
- 메뉴 [런타임] → [런타임 유형 변경]에서 **T4 GPU**를 고르면 학습이 훨씬 빠릅니다(CPU로도 됩니다).
- 설정을 바꾸면 **⑤부터 다시** 실행하세요.

In [ ]:
#@title ① [단계 2] 데이터 수집하기: crack.zip 불러오기
data_from = "직접 올리기 (crack.zip)" #@param ["직접 올리기 (crack.zip)", "구글 드라이브 (내 드라이브/data/crack)"]
import os, zipfile

def find_root(base):
    # train, test 폴더를 함께 가진 폴더를 찾는다
    for d, subdirs, _ in os.walk(base):
        if 'train' in subdirs and 'test' in subdirs:
            return d
    return None

if data_from.startswith('직접'):
    from google.colab import files
    print('[파일 선택] 버튼을 눌러 crack.zip을 고르세요.')
    up = files.upload()
    zip_name = [k for k in up if k.endswith('.zip')][0]
    zipfile.ZipFile(zip_name).extractall('/content/crack_data')
    root = find_root('/content/crack_data')
else:
    from google.colab import drive
    drive.mount('/content/gdrive')
    root = find_root('/content/gdrive/My Drive/data/crack')

assert root, 'train, test 폴더를 찾지 못했습니다. crack.zip을 다시 확인하세요.'
os.chdir(root)
for part in ['train', 'test']:
    for label in sorted(os.listdir(part)):
        if os.path.isdir(os.path.join(part, label)):
            print(f'{part}/{label}: {len(os.listdir(os.path.join(part, label)))}장')

In [ ]:
#@title ② 데이터 살펴보기: 위는 균열 없음(Negative), 아래는 균열 있음(Positive)
import matplotlib.pyplot as plt
from PIL import Image

plt.figure(figsize=(12, 4))
for row, label in enumerate(['Negative', 'Positive']):
    names = sorted(os.listdir(f'train/{label}'))[:6]
    for i, name in enumerate(names):
        plt.subplot(2, 6, row * 6 + i + 1)
        plt.imshow(Image.open(f'train/{label}/{name}'))
        plt.title(label, fontsize=9)
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#@title ③ [단계 2] 데이터 전처리하기: ImageDataGenerator (교과서 150쪽)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 픽셀값을 0~1 사이로 바꾸고, 크기를 64x64로 맞춘다
train_datagen = ImageDataGenerator(rescale=1./255)
training_set = train_datagen.flow_from_directory('train',
                                                 target_size=(64, 64),     # 이미지 크기 조정
                                                 batch_size=32,            # 한 번에 사용하는 이미지 수
                                                 shuffle=True,             # 훈련 이미지 순서 섞기
                                                 class_mode='categorical') # 다중 클래스 분류
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('test',
                                            target_size=(64, 64),
                                            shuffle=False,                 # 테스트 이미지 순서는 섞지 않기
                                            class_mode='categorical')
print('클래스 번호:', training_set.class_indices)

In [ ]:
#@title ④ [단계 3] 이미지 특징 추출하기: 사전 훈련된 VGG16 불러오기 (교과서 151쪽)
from keras.applications.vgg16 import VGG16

# include_top=False: 완전 연결 계층은 빼고 합성곱 층과 풀링 층만 가져온다
vgg = VGG16(include_top=False, weights='imagenet', input_shape=(64, 64, 3))
for layer in vgg.layers:
    layer.trainable = False        # 미리 학습한 가중치를 바꾸지 않고 그대로 쓴다

show_summary = False #@param {type:"boolean"}
n_conv = sum('conv' in l.name for l in vgg.layers)
n_pool = sum('pool' in l.name for l in vgg.layers)
print(f'VGG16에서 가져온 층: 합성곱 층 {n_conv}개, 풀링 층 {n_pool}개')
print('64x64 사진 한 장 → 특징맵', vgg.output_shape[1:], '=', vgg.output_shape[1] * vgg.output_shape[2] * vgg.output_shape[3], '개의 특징값')
if show_summary:
    vgg.summary()

### 예측해 보기
사진 300장으로 5번만 학습해도 **테스트 정확도**가 몇 %쯤 나올까요? 실행하기 전에 학습지에 예측값을 적어 두세요.

In [ ]:
#@title ⑤ [단계 3] 완전 연결 계층 설계하고 환경 설정하기 (교과서 151~152쪽)
hidden = 64 #@param [8, 16, 32, 64, 128, 256] {type:"raw"}
from keras.models import Sequential
from keras.layers import Dense, Flatten, Input

model = Sequential()
model.add(Input(shape=(64, 64, 3)))
model.add(vgg)                                   # VGG16으로 특징 추출
model.add(Flatten())                              # 1차원 배열로 바꾸기
model.add(Dense(hidden, activation='relu'))       # 은닉층
model.add(Dense(2, activation='softmax'))         # 출력층: Negative, Positive
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

trainable = sum(int(w.numpy().size) for w in model.trainable_weights)
print(f'은닉층 노드 {hidden}개로 모델을 만들었습니다.')
print(f'이번에 학습할 가중치 수: {trainable:,}개 (VGG16의 가중치 {vgg.count_params():,}개는 고정)')

In [ ]:
#@title ⑥ [단계 3] 신경망 모델 학습하기 (교과서 152쪽)
epochs = 5 #@param {type:"slider", min:1, max:15, step:1}
history = model.fit(training_set, epochs=epochs)

plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1); plt.plot(range(1, epochs + 1), history.history['accuracy'], 'o-'); plt.title('accuracy'); plt.xlabel('epoch')
plt.subplot(1, 2, 2); plt.plot(range(1, epochs + 1), history.history['loss'], 'o-', color='tab:red'); plt.title('loss'); plt.xlabel('epoch')
plt.tight_layout(); plt.show()

In [ ]:
#@title ⑦ [단계 4] 신경망 모델 평가하기 (교과서 152쪽)
loss, acc = model.evaluate(test_set, verbose=0)
print(f'테스트 데이터 {test_set.samples}장으로 평가한 정확도: {acc * 100:.1f}%  (손실 {loss:.3f})')

In [ ]:
#@title ⑧ [단계 4] 예측하고 혼동 행렬 그리기 (교과서 153쪽)
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

pred = model.predict(test_set, verbose=0)
Predicted = np.argmax(pred, axis=1)       # 가장 큰 확률의 클래스 번호
Actual = test_set.labels                  # 실제 정답
conf = confusion_matrix(Actual, Predicted)

plt.figure(figsize=(5, 4))
sns.heatmap(conf, annot=True, cmap='BuPu', fmt='d')
plt.title('Crack Classification'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()
names = {v: k for k, v in test_set.class_indices.items()}
for a in range(2):
    for p in range(2):
        if a != p and conf[a, p]:
            print(f'실제 {names[a]}인데 {names[p]}(으)로 잘못 예측: {conf[a, p]}장')

In [ ]:
#@title ⑨ 틀린 사진 직접 보기
wrong = np.where(Predicted != Actual)[0]
print('틀린 사진 수:', len(wrong))
if len(wrong):
    plt.figure(figsize=(12, 3))
    for i, k in enumerate(wrong[:8]):
        plt.subplot(1, 8, i + 1)
        plt.imshow(Image.open(test_set.filepaths[k]))
        plt.title(f'real {names[Actual[k]][:3]}\npred {names[Predicted[k]][:3]}', fontsize=8)
        plt.axis('off')
    plt.show()

### 기록하기
- ⑦의 테스트 정확도와 ⑧의 혼동 행렬 결과를 학습지에 적어 보세요.
- ⑤의 은닉층 노드 수나 ⑥의 학습 횟수를 바꿔 보고, 결과가 어떻게 달라지는지 적어 보세요.
- 모델이 틀린 사진들에는 어떤 공통점이 있나요?

In [ ]:
#@title ⑩ 내가 찍은 사진으로 예측하기
from google.colab import files
print('도로, 벽, 보도블록 사진을 올려 보세요.')
up = files.upload()
for name in up:
    img = Image.open(name).convert('RGB')
    x = np.array(img.resize((64, 64))) / 255.0
    p = model.predict(x[None], verbose=0)[0]
    plt.figure(figsize=(3, 3)); plt.imshow(img); plt.axis('off'); plt.show()
    print(f'{name}: 균열 없음 {p[test_set.class_indices["Negative"]] * 100:.1f}% / 균열 있음 {p[test_set.class_indices["Positive"]] * 100:.1f}%')